# Import Library

In [ ]:
from nltk.probability import FreqDist
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

import os
import spacy
import pickle
import pandas as pd
import numpy as np
import nltk

In [ ]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
spacy.cli.download("en_core_web_sm")

# Setting Variables

In [ ]:
stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()
eng_stopwords = set(stopwords.words("english"))

# Read Data

In [ ]:
dataset = pd.read_csv("./Dataset/imdb-movies-dataset.csv")
dataset.head(5)

# Simple Data Checking and Cleaning

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset = dataset.dropna()

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset["Sentiment"] = dataset["Rating"].apply(lambda x : "positive" if x > 5 else "negative")

In [ ]:
dataset["Sentiment"].value_counts()

# Data Preprocessing

In [ ]:
def preprocessing_text(sentence):

    # Tokenizing
    word_list = word_tokenize(sentence)
    word_list = (word.lower() for word in word_list)

    # Stopwords
    stopwordsWords = (token for token in word_list if token not in eng_stopwords)

    # No Punctuation
    no_punc = (token for token in stopwordsWords if token.isalpha())

    # Stemming
    stemmed = (stemmer.stem(word) for word in no_punc)

    # Lemmatized
    lemmatized = (lemmatizer.lemmatize(word) for word in stemmed)

    return lemmatized

# Frequency Distribution

In [ ]:
X = dataset["Review"]
Y = dataset["Sentiment"]

all_reviews = ' '.join(X)
all_tokens = preprocessing_text(all_reviews)

freq_dist = FreqDist(all_tokens)
print (freq_dist.most_common(10))

# Extract Features

In [ ]:
def extract_features(review):
    features = {}

    for word in freq_dist.keys():
        features[word] = (word in review)

    return features

In [ ]:
feature_sets = [(extract_features(preprocessing_text(review)), sentiment) for (review, sentiment) in zip (X,Y)]
from random import shuffle
shuffle(feature_sets)

In [ ]:
feature_sets

# Load and Train Model : Naive Bayes Classifier

In [ ]:
def load_and_save_model():

    train_count = int(len(feature_sets)*0.8)
    train_set = feature_sets[:train_count]
    test_set = feature_sets[train_count:]

    classifier = NaiveBayesClassifier.train(train_set)
    test_accuracy = accuracy(classifier, test_set)
    print (f"Model Accuracy : {test_accuracy}")
    classifier.show_most_informative_features(10)

    file = open("./model.pickle", "wb")
    pickle.dump(classifier, file)
    file.close()

    return classifier

def load_model():

    if os.path.exists("./model.pickle"):
        file = open("./model.pickle", "rb")
        classifier = pickle.load(file)
        classifier.show_most_informative_features(10)
        file.close()
        print ("Model Load Successfully")

    else:
        print ("Model Not Found. Training Model...")
        classifier = load_and_save_model()

    return classifier

# Word Embbeding Model

In [ ]:
def tf_idf(query):

    vectorizer = TfidfVectorizer(stop_words="english")
    tf_idf_matrix = vectorizer.fit_transform(dataset["Review"])

    query_vec = vectorizer.transform([query])

    similarity = cosine_similarity(tf_idf_matrix, query_vec).flatten()

    dataset["Similarity"] = similarity

    dataset_sorted = dataset.sort_values(by="Similarity", ascending=False)

    print ("Top 2 Movies Recommendation:")
    print (f"1. {dataset_sorted.iloc[0,0]} with similarity: {dataset_sorted["Similarity"].values[0]}")
    print (f"2. {dataset_sorted.iloc[1,0]} with similarity: {dataset_sorted["Similarity"].values[1]}")

# NER

In [ ]:
from collections import defaultdict
paragraph = ' '.join(dataset["Review"].head(500))

nlp = spacy.load("en_core_web_sm")

doc = nlp(paragraph)

categories = defaultdict(set)

for ent in doc.ents:
    label = ent.label_
    if label in ["LOC", "LANGUAGE"]:
        categories[label].add(ent.text)

# Menu Function

In [ ]:
my_review = "No Review"
my_category = "Unknown"

def menu_1(loaded_classifier):

    global my_review, my_category

    query = input ("Input Query : ")
    word = word_tokenize(query)

    if len(word) > 20:
        my_review = query
        preprocessing_texted = preprocessing_text(my_review)
        extracted = extract_features(preprocessing_texted)
        my_category = loaded_classifier.classify(extracted)
        print ("Review Saved Successfully")

    else:
        print ("Input Invalid!")

def menu_2():

    if my_review == "No Review":
        print ("Input Review First!")

    else:

        print ("Choose Model:")
        print ("1. TF-IDF\n2. Word2Vec\n3. N-Gram")
        
        chosen = input ("Input Model Number : ")

        if (chosen == '1'):
            tf_idf(my_review)
        elif (chosen == '2'):
            tf_idf(my_review)   
        elif (chosen == '3'):
            tf_idf(my_review)
        else:
            print ("Input Invalid!")

def menu_3():
    print ("NER:\n")
    for label, ent in categories.items():
        print (f"{label}: {', '.join(ent)}\n")

# Main Function

In [ ]:
def main_menu():
    loaded_classifier = load_model()

    while True:
        print ("Movie Recommendation Application Based on Review")
        print (f"Your Review: {my_review}")
        print (f"My Category: {my_category}")

        print ("1. Write your Review\n2. View Movie Recommendation\n3. View NER\n4. Exit")

        try:
            choice = input(">> ")
            if choice == '1':
                menu_1(loaded_classifier)
            elif (choice == '2'):
                menu_2()
            elif (choice == '3'):
                menu_3()
            elif (choice == '4'):
                print ("Exit Menu...")
                break
            else:
                print ("Input Invalid!")
        except ValueError:
            print ("Input Invalid, please enter a valid number!")

In [ ]:
main_menu()